### Model Subclassing & Custom Training Loop Using Reuters Dataset

In [58]:
#importing libraries

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import time

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, Softmax
from tensorflow.keras.utils import to_categorical

In [59]:
#defining the custom layers and model

class MyLayer(Layer):

    def __init__(self, units):
        super(MyLayer, self).__init__()
        self.units = units

    def build(self, input_shape):

        self.w = self.add_weight(shape=(input_shape[-1], self.units),
                    initializer="random_normal",
                    name="kernel")
        self.b = self.add_weight(shape=(self.units, ),
                    initializer="zeros",
                    name="bias")

    def call(self, inputs):
        return tf.matmul(inputs, self.w) + self.b


class MyDropout(Layer):

    def __init__(self, rate):
        super(MyDropout, self).__init__()
        self.rate = rate

    def call(self, inputs):
        return tf.nn.dropout(inputs, rate=self.rate)  


class MyModel(Model):

    def __init__(self, units_1, units_2, units_3):
        super(MyModel, self).__init__()
        self.layer_1 = MyLayer(units_1)  
        self.layer_2 = MyLayer(units_2)
        self.layer_3 = MyLayer(units_3)
        self.dropout_1 = MyDropout(0.5)
        self.dropout_2 = MyDropout(0.5)
        self.softmax = Softmax()

    def call(self, inputs):
        #define forward pass
        x = self.layer_1(inputs)
        x = tf.nn.relu(x)
        x = self.dropout_1(x)
        x = self.layer_2(x)
        x = tf.nn.relu(x)
        x = self.dropout_2(x)
        x = self.layer_3(x)
        x = tf.nn.relu(x)
        return self.softmax(x)


In [60]:
#instantiating the model object

model = MyModel(64, 64, 46)
print(model(tf.ones((1, 10000))))
model.summary()

tf.Tensor(
[[0.01704969 0.01818435 0.01533254 0.02185015 0.02464453 0.03345283
  0.01533254 0.05153361 0.02361255 0.03850476 0.0263548  0.01533254
  0.01533254 0.01533254 0.01533254 0.01865496 0.01951423 0.03792541
  0.02988537 0.01533254 0.01533254 0.01533254 0.01824822 0.02390843
  0.01673624 0.03119949 0.02393997 0.01533254 0.01533254 0.05411236
  0.01533254 0.01770049 0.05250737 0.01533254 0.01533254 0.01595436
  0.01533254 0.01573899 0.01533254 0.01683431 0.01533254 0.01533254
  0.01581008 0.0223241  0.01533254 0.0225001 ]], shape=(1, 46), dtype=float32)


Model: "my_model_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ my_layer_15 (MyLayer)           │ ?                      │       640,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_layer_16 (MyLayer)           │ ?                      │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_layer_17 (MyLayer)           │ ?                      │         2,990 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_dropout_10 (MyDropout)       │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_dropout_11 (MyDropout)       │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax_5 (Softmax)             │ ?                      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 647,214 (2.47 MB)

 Trainable params: 647,214 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

In [61]:
#loading the reuters dataset

from tensorflow.keras.datasets import reuters

(train_data, train_labels), (test_data, test_labels) = reuters.load_data(num_words=10000)

class_names = ['cocoa','grain','veg-oil','earn','acq','wheat','copper','housing','money-supply',
   'coffee','sugar','trade','reserves','ship','cotton','carcass','crude','nat-gas',
   'cpi','money-fx','interest','gnp','meal-feed','alum','oilseed','gold','tin',
   'strategic-metal','livestock','retail','ipi','iron-steel','rubber','heat','jobs',
   'lei','bop','zinc','orange','pet-chem','dlr','gas','silver','wpi','hog','lead']

d:\Tensorflow_Works\.venv\Lib\site-packages\numpy\lib\_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


In [62]:
#printing the class of first sample

print("Label: {}".format(class_names[train_labels[0]]))

Label: earn


In [63]:
#loading the reuters word index

word_to_index = reuters.get_word_index()
invert_word_index = dict([(value, key) for (key, value) in word_to_index.items()])
text_news = " ".join([invert_word_index.get(value-3, "?") for value in train_data[0]])
text_news 

'? ? ? said as a result of its december acquisition of space co it expects earnings per share in 1987 of 1 15 to 1 30 dlrs per share up from 70 cts in 1986 the company said pretax net should rise to nine to 10 mln dlrs from six mln dlrs in 1986 and rental operation revenues to 19 to 22 mln dlrs from 12 5 mln dlrs it said cash flow per share this year should be 2 50 to three dlrs reuter 3'

In [64]:
#defining a function that encodes the data into a "bag of words" representation

def bag_of_words(text_samples, elements=10000):
    output = np.zeros((len(text_samples), elements))
    for i, word in enumerate(text_samples):
        output[i, word] =1.
    return output

x_train = bag_of_words(train_data)
x_test = bag_of_words(test_data)

print("Shape of x_train:", x_train.shape)
print("Shape of x_test:", x_test.shape)


Shape of x_train: (8982, 10000)
Shape of x_test: (2246, 10000)


In [65]:
x_train[0]

array([0., 1., 1., ..., 0., 0., 0.], shape=(10000,))

In [76]:
#define the loss function and optimizer

loss_object = tf.keras.losses.SparseCategoricalCrossentropy()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)  

def loss(model, x, y, wd):
    kernel_variables=[]
    for l in model.layers:
        for w in l.weights:
            if "kernel" in w.name:
                kernel_variables.append(w)
    wd_penalty = wd * tf.reduce_sum([tf.reduce_sum(tf.square(k)) for k in kernel_variables])
    y_ = model(x, training=True)
    base_loss = loss_object(y_true=y, y_pred=y_)
    return  base_loss + wd_penalty 

             

In [77]:
#training the model

def grad(model, inputs, targets, wd):
    with tf.GradientTape() as tape:
        loss_value = loss(model, inputs, targets, wd)
    return (loss_value, tape.gradient(loss_value, model.trainable_variables) )   

In [83]:
#implementing the training loop

start_time = time.time()

train_dataset = tf.data.Dataset.from_tensor_slices((x_train, train_labels))
train_dataset = train_dataset.batch(32)

#keep results for plotting
train_loss_results = []
train_accuracy_results = []

num_epochs= 10
weight_decay = 0.005

for epoch in range(num_epochs):
    epoch_loss_avg = tf.keras.metrics.Mean()
    epoch_accuracy = tf.keras.metrics.CategoricalAccuracy()

    #training loop
    for x, y in train_dataset:
        #optimize the model
        loss_value, grads = grad(model, x, y, weight_decay)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

        #compute current loss
        epoch_loss_avg(loss_value)
        #compare predicted value to actual label
        epoch_accuracy(to_categorical(y.numpy()), model(x))

    #end epoch
    train_loss_results.append(epoch_loss_avg.result())
    train_accuracy_results.append(epoch_accuracy.result())

    print("Epoch {:03d}: Loss: {:.3f}, Accuracy: {:.3%}".format(epoch,
                                                    epoch_loss_avg.result(),
                                                    epoch_accuracy.result() ))

print("Duraction : {:.3f}".format(time.time() - start_time))

Epoch 000: Loss: 3.286, Accuracy: 48.085%
Epoch 001: Loss: 1.953, Accuracy: 60.376%
Epoch 002: Loss: 1.881, Accuracy: 65.308%
Epoch 003: Loss: 1.843, Accuracy: 67.724%
Epoch 004: Loss: 1.814, Accuracy: 68.337%
Epoch 005: Loss: 1.808, Accuracy: 69.572%
Epoch 006: Loss: 1.790, Accuracy: 69.595%
Epoch 007: Loss: 1.787, Accuracy: 69.996%
Epoch 008: Loss: 1.768, Accuracy: 70.497%
Epoch 009: Loss: 1.769, Accuracy: 70.920%
Duraction : 196.134
